# 动态规划（旅行商问题）

在本 notebook 中，我们将探索使用动态规划求解旅行商问题。

## 环境配置说明：
### 如果在本地运行：
#### 首次配置
* 安装Anaconda：访问 https://www.anaconda.com/products/distribution
* 创建conda环境：`conda create -n cs131232 python=3.8`
  * 详见 https://conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html
* 激活环境：`conda activate cs131232`
* 安装pip：`conda install pip`
* 安装Jupyter：`conda install jupyter`
#### 每次启动
* 确保激活环境：`conda activate cs131232`
* 启动Jupyter：`jupyter notebook` 或 `jupyter lab`
* 运行以下单元格**并在需要时重启内核**

In [ ]:
# Install dependencies
!pip install -r requirements.txt --quiet

In [ ]:
import otter
assert (otter.__version__ >= "5.5.0"), "Please reinstall the requirements and restart your kernel."

grader = otter.Notebook("dp-tsp.ipynb")
import numpy.random as random
from networkx import Graph, draw
import string
import pylev
import tqdm
import time
import pickle 

from autograder_utils import validate_tour, handle_timeout, profile

test_data = pickle.load(open("public_data.pkl", "rb"))

rng_seed = 0

## Q1：旅行商问题的动态规划

现在，我们将实现旅行商问题（TSP）的动态规划算法。即使对于中等规模的测试用例，暴力解法也会慢得令人绝望，但我们可以使用动态规划在稍微合理一些（但仍然是指数级）的时间内得到解。

与之前的问题一样，我们希望返回实际的巡回路线，而不是巡回路线代价。我们可以再次采用相同的做法，通过在子问题数组中回溯来重建这条巡回路线。

### 表示子问题

如果我们把集合作为子问题的一个参数，就不能直接用二维数组来存储子问题。有两种常见的变通方法：

#### 1. 子问题字典

你可以把子问题存储在字典中，键的形式为 `(S, i)`，其中 `i` 表示返回家之前最后访问的城市，`S` 是到目前为止已访问城市的集合。

为了让这种方法可行，你需要确保键是可哈希的。一种方法是使用 Python 内置的 `frozenset` 类来表示 `S`。`frozenset` 是内置类型，因此无需额外导入即可使用；它的工作方式与普通集合一样，只是不可变（因而可哈希）。你可以在这里了解更多关于 `frozenset` 的内容：https://docs.python.org/3/library/stdtypes.html#frozenset。

#### 2. 位掩码

除了哈希集合，我们实际上仍然可以使用二维数组来存储子问题，其中 `S` 用一个 $n$ 位无符号整数表示；当且仅当第 $i$ 个城市属于已访问城市集合时，`S` 的第 $i$ 位被置为 1。由于 `S` 是整数，我们可以用它来索引二维数组。

位掩码方法通常比 `frozenset` 方法快大约一倍，并且内存效率高得多，但只要实现正确，两种方法都能通过自动评分器。

### 实现提示：

和之前一样，在每一步存储整条巡回路线会占用过多内存，并会导致自动评分器失败。相反，可以考虑维护一个单独的字典或数组，它存储较少的信息，但仍能帮助你重建巡回路线（“DP DAG 中的最短路径”这一思路能在这里帮上忙吗？）。

**注意索引！** 书中的算法假设城市编号为 $1, \dots, n$ —— 如果你要索引 Python 列表，是否需要调整索引？

注意子问题的求解顺序！我们需要确保：每当求解一个子问题时，它所依赖的所有子问题都已经求解完毕。

图不一定是完全图！如果不存在可行的巡回路线，请返回空列表。

_分值：_ 50

### 图辅助函数

与之前的作业一样，我们使用加权邻接表来表示图。我们将沿用与之前类似的格式，不同之处在于 `graph[u]` 现在是一个哈希映射，而不是一组键值对列表。
**在本作业中，图是无向图**，因此如果节点 `u` 和 `v` 之间存在一条权重为 `w` 的（无向）边，那么 `graph[u]` 包含键 `v`，值为 `w`，并且 `graph[v]` 包含键 `u`，值为 `w`。

我们提供以下代码来帮助你测试你的实现。

In [ ]:
def generate_adj_list(n, edge_list: list[tuple[int]]) -> list[dict[int, int]]:
    """
    args:
        n:int = number of nodes in the graph. The nodes are labelled with integers 0 through n-1
        edge_list:list[tuple[int,int,int]] = edge list where each tuple (u,v,w) represents the 
        undirected and weighted edge (u,v,w) in the graph
    return:
        A list[dict[int, int]] representing the adjacency list 
    """
    adj_list = [{} for _ in range(n)] 
    for u, v, w in sorted(edge_list):
        adj_list[u][v] = w
        # undirected edges
        adj_list[v][u] = w
    return adj_list

def draw_graph(adj_list: list[dict[int, int]]):
    """Utility method for visualizing graphs

    args:
        adj_list: list[dict[int, int]] = adjacency list representation of the graph generated by generate_adj_list
    """
    G = Graph()
    for u in range(len(adj_list)):
        for v, w in adj_list[u].items():
            G.add_edge(u, v, weight=w)
    draw(G, with_labels=True)

In [ ]:
def tsp_dp(adj_list):
    """Compute the exact solution to the TSP using dynamic programming and returns the optimal path.

    Args:
        dist_arr: Weighted undirected graph represented as an adjacency list. 

    Returns:
        List[int]: A list of city indices representing the optimal path.
    """
    n = len(adj_list)
    if n == 0:
        return []

    # Held-Karp with city 0 fixed as the start of the tour.  dp[mask][last]
    # is the cheapest path that starts at 0, visits exactly mask, and ends at
    # last; parent[] stores only the predecessor needed to rebuild the tour.
    start = 0
    full_mask = (1 << n) - 1
    dp = [None] * (1 << n)
    parent = [None] * (1 << n)
    dp[1] = {start: 0}
    parent[1] = {start: -1}

    # Every mask containing the start is odd.
    for mask in range(1, 1 << n, 2):
        layer = dp[mask]
        if not layer:
            continue
        for last, cost in layer.items():
            for nxt, weight in adj_list[last].items():
                bit = 1 << nxt
                if mask & bit:
                    continue
                next_mask = mask | bit
                next_cost = cost + weight
                next_layer = dp[next_mask]
                if next_layer is None:
                    next_layer = {}
                    dp[next_mask] = next_layer
                    parent[next_mask] = {}
                old = next_layer.get(nxt)
                if old is None or next_cost < old:
                    next_layer[nxt] = next_cost
                    parent[next_mask][nxt] = last

    best_cost = float("inf")
    best_last = -1
    full_layer = dp[full_mask]
    if full_layer:
        for last, cost in full_layer.items():
            closing = adj_list[last].get(start)
            if closing is not None and cost + closing < best_cost:
                best_cost = cost + closing
                best_last = last

    if best_last == -1:
        return []

    tour = []
    mask = full_mask
    last = best_last
    while last != -1:
        tour.append(last)
        previous = parent[mask][last]
        mask ^= 1 << last
        last = previous
    tour.reverse()
    return tour


In [ ]:
grader.check("TSP")

### 调试

为方便起见，这里粘贴了 Otter 测试的简化版本。调试时，你可以随意添加打印语句或断言。

In [ ]:
# tests on very small cases
for adj_list, expected_distance in tqdm.tqdm(test_data['TSP-1']):
    result = tsp_dp(adj_list)

    if expected_distance < 0: 
        # no tour is possible
        
        assert result == [], "You returned a tour when no tour is possible"
    else:
        assert set(result) == set(range(len(adj_list))), f"Your output does not visit all cities"
        student_length = validate_tour(result, adj_list)
        assert student_length >= 0, f"Your output is not a valid tour"
        assert student_length == expected_distance, f"Your output is not a minimum distance tour"

## 提交

确保在运行下面的单元格之前，你已经按顺序运行了笔记本中的所有单元格，以便所有图像/图表都能在输出中显示。下面的单元格将为你生成一个用于提交的压缩文件。

In [ ]:
grader.export(pdf=False, force_save=True, run_tests=True)